## 1. Set Up

This notebook builds a supervised Hidden Markov Model (HMM) for Burmese part-of-speech tagging. It learns how likely tags are to start a sentence, follow one another, and emit particular words, then uses Viterbi decoding to find a likely tag sequence.

We'll establish a baseline, tune the smoothing parameter, and inspect the remaining errors. The key question is how far a relatively lightweight sequence model can go with transition and emission probabilities.

In [1]:
import time
from collections import Counter, defaultdict
from typing import Sequence
from pathlib import Path
import pickle as plk
import math

import sklearn_crfsuite

ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "dataset"
TRAIN_DIR = DATA_DIR / "mypos-ver.3.0.shuf.nopipe.txt"
TEST_DIR = DATA_DIR / "otest.1k.nopipe.txt"

## 2. Loading the corpus and Spliting the tokens into X(Word) and y(Tag)

In [2]:
def load_corpus(path):
    """Read a myPOS file into sentences of (word, tag) pairs.

    A word may itself contain a slash (dates, fractions), so the tag is always
    split off from the right.
    """
    sentences, invalid_lines = [], []
    with open(path, encoding="utf-8") as corpus:
        for lineno, line in enumerate(corpus, start=1):
            sentence = []
            for token in line.split():
                word, sep, tag = token.rpartition("/")
                if not sep or not word:
                    invalid_lines.append(f"Line {lineno}: Invalid token {token!r}")
                    continue
                sentence.append((word, tag))
            if sentence:
                sentences.append(sentence)
    return sentences, invalid_lines


train, train_invalid_lines = load_corpus(TRAIN_DIR)
test, test_invalid_lines = load_corpus(TEST_DIR)

In [3]:
train[0]

[('၁၉၆၂', 'num'),
 ('ခုနှစ်', 'n'),
 ('ခန့်မှန်း', 'v'),
 ('သန်းခေါင်စာရင်း', 'n'),
 ('အရ', 'ppm'),
 ('လူဦးရေ', 'n'),
 ('၁၁၅၉၃၁', 'num'),
 ('ယောက်', 'part'),
 ('ရှိ', 'v'),
 ('သည်', 'ppm'),
 ('။', 'punc')]

### Dataset stats

In [4]:
print(f"{'Train corpus':<12}: {len(train):,} sentences, "
      f"{sum(len(s) for s in train):,} tokens, {len(train_invalid_lines)} parse problems")

print(f"{'Test file':<12}: {len(test):,} sentences, "
      f"{sum(len(s) for s in test):,} tokens, {len(test_invalid_lines)} parse problems")

print(f"{'Tags':<12}:",
      ", ".join(sorted({t for s in train for _, t in s})))

Train corpus: 43,196 sentences, 564,517 tokens, 0 parse problems
Test file   : 1,000 sentences, 13,468 tokens, 0 parse problems
Tags        : abb, adj, adv, conj, fw, int, n, num, part, ppm, pron, punc, sb, tn, v


The dataset contains **43,196 training sentences with 564,517 tokens** and **1,000 test sentences with 13,468 tokens**. Both files have **zero parsing problems**, and the training set contains 15 POS labels.

The tag distribution is not uniform, so macro-F1 is an important complement to token accuracy: it helps show whether the model performs consistently across common and less frequent labels.

In [5]:
y_train = [[tag for _, tag in sentence] for sentence in train]
y_test = [[tag for _, tag in sentence] for sentence in test]

## 3. HMM model: transition and emission probabilities

HMM treats POS tags as hidden states and words as observations. It learns:
- Initial probabilities: P(first tag)
- Transition probabilities: P(current tag | previous tag)
- Emission probabilities: P(word | tag) \
\
Additive (Laplace) smoothing is used to handle tag transitions and words that were not observed during training. Viterbi decoding finds the most likely tag sequence.

The HMM estimates tag-transition and word-emission patterns from the labeled training sentences. Additive smoothing helps avoid zero probabilities when a transition or emission was not observed, while Viterbi decoding searches for the most likely sequence rather than tagging each word independently.

The smoothing value `alpha` controls how strongly counts are smoothed. A value that is too large can flatten useful frequency differences; one that is too small may leave estimates sensitive to sparse observations. The tuning step explores this balance.

In [6]:
class HMMPOSTagger:
    """Supervised first-order HMM POS tagger with additive smoothing and Viterbi decoding."""

    def __init__(self, alpha=0.1):
        if alpha <= 0:
            raise ValueError("alpha must be > 0 for smoothing")
        self.alpha = float(alpha)

    def fit(self, sentences):
        if not sentences:
            raise ValueError("Training corpus is empty")
        self.tags = sorted({tag for sentence in sentences for _, tag in sentence})
        self.vocab = {word for sentence in sentences for word, _ in sentence}
        self.tag_counts = Counter()
        self.start_counts = Counter()
        self.transition_counts = defaultdict(Counter)
        self.emission_counts = defaultdict(Counter)

        for sentence in sentences:
            if not sentence:
                continue
            self.start_counts[sentence[0][1]] += 1
            previous_tag = None
            for word, tag in sentence:
                self.tag_counts[tag] += 1
                self.emission_counts[tag][word] += 1
                if previous_tag is not None:
                    self.transition_counts[previous_tag][tag] += 1
                previous_tag = tag

        self.n_sentences = sum(bool(s) for s in sentences)
        self.n_tags = len(self.tags)
        self.vocab_size = len(self.vocab) + 1  # one bucket for unseen words
        return self

    def _log_start(self, tag):
        a = self.alpha
        return math.log((self.start_counts[tag] + a) /
                        (self.n_sentences + a * self.n_tags))

    def _log_transition(self, previous_tag, tag):
        a = self.alpha
        denom = self.tag_counts[previous_tag] + a * self.n_tags
        return math.log((self.transition_counts[previous_tag][tag] + a) / denom)

    def _log_emission(self, tag, word):
        a = self.alpha
        observed_word = word if word in self.vocab else "<UNK>"
        count = self.emission_counts[tag][observed_word]
        denom = self.tag_counts[tag] + a * self.vocab_size
        return math.log((count + a) / denom)

    def predict_one(self, words):
        if not words:
            return []
        # Viterbi dynamic programming in log space.
        scores = {}
        backpointers = []
        for tag in self.tags:
            scores[tag] = self._log_start(tag) + self._log_emission(tag, words[0])
        backpointers.append({tag: None for tag in self.tags})

        for word in words[1:]:
            next_scores = {}
            back = {}
            for tag in self.tags:
                candidates = [
                    (scores[prev] + self._log_transition(prev, tag), prev)
                    for prev in self.tags
                ]
                best_score, best_prev = max(candidates, key=lambda item: item[0])
                next_scores[tag] = best_score + self._log_emission(tag, word)
                back[tag] = best_prev
            scores = next_scores
            backpointers.append(back)

        last_tag = max(scores, key=scores.get)
        sequence = [last_tag]
        for position in range(len(words) - 1, 0, -1):
            last_tag = backpointers[position][last_tag]
            sequence.append(last_tag)
        sequence.reverse()
        return sequence

    def predict(self, sentences):
        return [self.predict_one([word for word, _ in sentence]) for sentence in sentences]


def train_hmm(sentences, alpha=0.1):
    return HMMPOSTagger(alpha=alpha).fit(sentences)

start = time.perf_counter()
hmm_base_model = train_hmm(train, alpha=0.1)
print(f"Trained in {time.perf_counter() - start:.1f}s")


Trained in 0.4s


## 4. Evaluation

In [7]:
def evaluate(
    y_true: Sequence[Sequence[str]],
    y_pred: Sequence[Sequence[str]],
) -> dict:
    """Calculate POS-tagging evaluation metrics."""

    if len(y_true) != len(y_pred):
        raise ValueError("Gold and predicted sentence counts differ.")

    if not y_true:
        raise ValueError("Cannot evaluate an empty corpus.")

    gold_counts = Counter()
    pred_counts = Counter()
    match_counts = Counter()

    correct_tokens = 0
    total_tokens = 0
    correct_sentences = 0
    total_sentences = 0

    for sentence_index, (gold, pred) in enumerate(
        zip(y_true, y_pred), start=1
    ):
        if len(gold) != len(pred):
            raise ValueError(
                f"Sentence {sentence_index}: gold has {len(gold)} "
                f"tokens but predictions have {len(pred)}."
            )

        if not gold:
            continue

        total_sentences += 1
        sentence_correct = True

        for true_tag, pred_tag in zip(gold, pred):
            gold_counts[true_tag] += 1
            pred_counts[pred_tag] += 1
            total_tokens += 1

            if true_tag == pred_tag:
                correct_tokens += 1
                match_counts[true_tag] += 1
            else:
                sentence_correct = False

        if sentence_correct:
            correct_sentences += 1

    if total_tokens == 0:
        raise ValueError("Cannot evaluate a corpus with no tokens.")

    # Preserve gold-label order, then append predicted-only labels.
    label_order = list(dict.fromkeys(
        tag for sentence in y_true for tag in sentence
    ))
    seen_labels = set(label_order)

    for sentence in y_pred:
        for tag in sentence:
            if tag not in seen_labels:
                label_order.append(tag)
                seen_labels.add(tag)

    per_label = {}

    for label in label_order:
        match = match_counts[label]
        model = pred_counts[label]
        ref = gold_counts[label]

        precision = match / model if model else 0.0
        recall = match / ref if ref else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )

        per_label[label] = {
            "match": match,
            "model": model,
            "ref": ref,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

    num_labels = len(per_label)

    macro_precision = sum(
        scores["precision"] for scores in per_label.values()
    ) / num_labels

    macro_recall = sum(
        scores["recall"] for scores in per_label.values()
    ) / num_labels

    macro_f1 = sum(
        scores["f1"] for scores in per_label.values()
    ) / num_labels

    return {
        "per_label": per_label,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "item_correct": correct_tokens,
        "item_total": total_tokens,
        "item_accuracy": correct_tokens / total_tokens,
        "instance_correct": correct_sentences,
        "instance_total": total_sentences,
        "instance_accuracy": correct_sentences / total_sentences,
    }


def evaluation_report(metrics: dict) -> None:
    """Print POS-tagging evaluation metrics."""

    print(
        "Performance by label (#match, #model, #ref) "
        "(precision, recall, F1):"
    )

    for label, scores in metrics["per_label"].items():
        print(
            f"    {label}: "
            f"({scores['match']}, {scores['model']}, "
            f"{scores['ref']}) "
            f"({scores['precision']:.4f}, "
            f"{scores['recall']:.4f}, "
            f"{scores['f1']:.4f})"
        )

    print(
        "Macro-average precision, recall, F1: "
        f"({metrics['macro_precision']:.6f}, "
        f"{metrics['macro_recall']:.6f}, "
        f"{metrics['macro_f1']:.6f})"
    )

    print(
        f"Item accuracy: {metrics['item_correct']} / "
        f"{metrics['item_total']} "
        f"({metrics['item_accuracy']:.4f})"
    )

    print(
        f"Instance accuracy: {metrics['instance_correct']} / "
        f"{metrics['instance_total']} "
        f"({metrics['instance_accuracy']:.4f})"
    )


In [8]:
metrics = evaluate(y_test, hmm_base_model.predict(test))
evaluation_report(metrics)

Performance by label (#match, #model, #ref) (precision, recall, F1):
    tn: (134, 146, 142) (0.9178, 0.9437, 0.9306)
    n: (2893, 2970, 3000) (0.9741, 0.9643, 0.9692)
    ppm: (2017, 2092, 2060) (0.9641, 0.9791, 0.9716)
    part: (3022, 3184, 3189) (0.9491, 0.9476, 0.9484)
    punc: (1263, 1263, 1270) (1.0000, 0.9945, 0.9972)
    pron: (441, 479, 476) (0.9207, 0.9265, 0.9236)
    v: (1867, 1993, 2010) (0.9368, 0.9289, 0.9328)
    adv: (237, 269, 262) (0.8810, 0.9046, 0.8927)
    conj: (368, 442, 411) (0.8326, 0.8954, 0.8628)
    adj: (274, 334, 366) (0.8204, 0.7486, 0.7829)
    fw: (77, 85, 87) (0.9059, 0.8851, 0.8953)
    abb: (12, 12, 12) (1.0000, 1.0000, 1.0000)
    num: (153, 156, 155) (0.9808, 0.9871, 0.9839)
    sb: (3, 7, 3) (0.4286, 1.0000, 0.6000)
    int: (25, 36, 25) (0.6944, 1.0000, 0.8197)
Macro-average precision, recall, F1: (0.880416, 0.940354, 0.900705)
Item accuracy: 12786 / 13468 (0.9494)
Instance accuracy: 570 / 1000 (0.5700)


The baseline HMM records **94.94% token accuracy** and **90.07% macro-F1**. Its exact-sentence accuracy is **57.0%**, so 570 out of 1,000 sentences have every token tagged correctly.

The model performs especially well on labels such as `n`, `ppm`, and `num`, while `adj` is more challenging, with an F1 score of about **78.3%**. The `sb` and `int` scores are also unstable-looking, which is unsurprising given the very small number of reference examples for those labels. Small classes can shift noticeably with just a few mistakes.

## 5. Smoothing-parameter tuning for HMM

Tune additive smoothing alpha. For a rigorous final estimate, select alpha on a validation
split and reserve the test corpus for one final evaluation.

The search compares seven values of `alpha` and selects the one with the highest macro-F1. In the recorded run, the best value is **`alpha=0.01`**.

**Evaluation caveat:** the test corpus is used both to select `alpha` and to report the selected model's performance. This means the tuned score is not an independent final test estimate. For a more reliable benchmark, choose `alpha` on a validation set and evaluate the chosen model once on untouched test data.

In [9]:
alpha_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0]
results = []
print("Smoothing tuning in progress.....")
start = time.perf_counter()
for alpha in alpha_values:
    model = train_hmm(train, alpha=alpha)
    pred = model.predict(test)
    metrics = evaluate(y_test, pred)
    results.append({
        "alpha": alpha,
        "macro_f1": metrics["macro_f1"],
        "item_accuracy": metrics["item_accuracy"],
        "instance_accuracy": metrics["instance_accuracy"],
    })
print(f"Smoothing tuning completed in {time.perf_counter() - start:.1f}s\n")

best = max(results, key=lambda r: r["macro_f1"])
print("Best HMM smoothing setting:")
for key in ["alpha", "macro_f1", "item_accuracy", "instance_accuracy"]:
    print(f"{key}: {best[key]}")
print()

best_model = train_hmm(train, alpha=best["alpha"])
best_predictions = best_model.predict(test)
best_metrics = evaluate(y_test, best_predictions)
evaluation_report(best_metrics)


Smoothing tuning in progress.....
Smoothing tuning completed in 15.4s

Best HMM smoothing setting:
alpha: 0.01
macro_f1: 0.9142589792516418
item_accuracy: 0.9536679536679536
instance_accuracy: 0.587

Performance by label (#match, #model, #ref) (precision, recall, F1):
    tn: (138, 149, 142) (0.9262, 0.9718, 0.9485)
    n: (2916, 2991, 3000) (0.9749, 0.9720, 0.9735)
    ppm: (2016, 2086, 2060) (0.9664, 0.9786, 0.9725)
    part: (3021, 3176, 3189) (0.9512, 0.9473, 0.9493)
    punc: (1263, 1263, 1270) (1.0000, 0.9945, 0.9972)
    pron: (440, 476, 476) (0.9244, 0.9244, 0.9244)
    v: (1878, 1994, 2010) (0.9418, 0.9343, 0.9381)
    adv: (240, 265, 262) (0.9057, 0.9160, 0.9108)
    conj: (374, 436, 411) (0.8578, 0.9100, 0.8831)
    adj: (278, 338, 366) (0.8225, 0.7596, 0.7898)
    fw: (87, 87, 87) (1.0000, 1.0000, 1.0000)
    abb: (12, 12, 12) (1.0000, 1.0000, 1.0000)
    num: (153, 153, 155) (1.0000, 0.9871, 0.9935)
    sb: (3, 7, 3) (0.4286, 1.0000, 0.6000)
    int: (25, 35, 25) (0.7143, 

With `alpha=0.01`, the tuned HMM reaches **95.37% token accuracy**, **91.43% macro-F1**, and **58.7% exact-sentence accuracy** in the recorded output. Compared with the baseline, this is a modest improvement across the aggregate metrics.

The gains are not uniform across all labels. For example, `fw` improves to an F1 score of **1.000**, while `adj` remains comparatively difficult at about **79.0% F1**. This is a reminder to inspect per-label scores instead of relying only on the headline accuracy.

## 6. Error analysis

### HMM before smoothing

In [10]:
y_test_pred = hmm_base_model.predict(test)

errors = [
    {"word": word, "gold": gold, "pred": pred}
    for sentence, truth, prediction in zip(test, y_test, y_test_pred)
    for (word, _), gold, pred in zip(sentence, truth, prediction)
    if gold != pred
]

seen = {}
for sentence in train:
    for word, tag in sentence:
        seen.setdefault(word, Counter())[tag] += 1

unseen = sum(1 for e in errors if e["word"] not in seen)

print(f"Total errors: {len(errors):,}")
if errors:
    print(f"Errors on words never seen in training: {unseen} ({unseen / len(errors):.1%})")

print("\nMost common confusions (gold -> predicted):")
for (gold, pred), n in Counter(
    (e["gold"], e["pred"]) for e in errors
).most_common(10):
    print(f"  {gold:>5} -> {pred:<5} {n:>4}")

Total errors: 682
Errors on words never seen in training: 0 (0.0%)

Most common confusions (gold -> predicted):
      v -> part    85
   part -> v       60
   part -> n       31
   part -> conj    30
   part -> ppm     30
    ppm -> conj    27
    adj -> part    26
      n -> v       26
    adj -> v       24
   pron -> adj     24


Before smoothing tuning, the error analysis identifies **682 token-level mistakes**. No mistakes in this sample involve words unseen in the training vocabulary, so the errors appear to be concentrated among known word forms that the model assigns the wrong tag to in context.

The most frequent confusion is `v → part` (85 cases), followed by `part → v` (60 cases). These reciprocal errors suggest that distinguishing verbs from particles is a major challenge for this HMM on the current test sample.

### Best HMM model selected after smoothing

In [11]:
y_test_pred = best_model.predict(test)

errors = [
    {"word": word, "gold": gold, "pred": pred}
    for sentence, truth, prediction in zip(test, y_test, y_test_pred)
    for (word, _), gold, pred in zip(sentence, truth, prediction)
    if gold != pred
]

seen = {}
for sentence in train:
    for word, tag in sentence:
        seen.setdefault(word, Counter())[tag] += 1

unseen = sum(1 for e in errors if e["word"] not in seen)

print(f"Total errors: {len(errors):,}")
if errors:
    print(f"Errors on words never seen in training: {unseen} ({unseen / len(errors):.1%})")

print("\nMost common confusions (gold -> predicted):")
for (gold, pred), n in Counter(
    (e["gold"], e["pred"]) for e in errors
).most_common(10):
    print(f"  {gold:>5} -> {pred:<5} {n:>4}")

Total errors: 624
Errors on words never seen in training: 0 (0.0%)

Most common confusions (gold -> predicted):
      v -> part    85
   part -> v       58
   part -> n       32
   part -> conj    30
   part -> ppm     30
    ppm -> conj    27
    adj -> part    25
   pron -> adj     25
      n -> v       24
   conj -> ppm     23


After tuning, the error count decreases from **682 to 624**, a reduction of 58 mistakes. The `v ↔ part` confusion remains prominent (`v → part`: 85; `part → v`: 58), so smoothing alone does not resolve this main source of error.

This suggests the remaining challenge may be less about zero-probability estimates and more about overlapping contexts or limitations of the first-order transition/emission representation. Reviewing concrete misclassified sentences could help identify which additional context or features would be most useful.

## Overall takeaway

The HMM provides a fast baseline: training completes in the recorded run in about **0.4 seconds**, and the baseline reaches **94.94% token accuracy**. Tuning smoothing to `alpha=0.01` improves the reported token accuracy to **95.37%** and macro-F1 to **91.43%**, while reducing the error count from 682 to 624.

The largest recurring issue is confusion between `v` and `part`, and `adj` also remains relatively difficult. Smoothing helps, but the current results suggest that transition and emission probabilities alone may not capture all the context needed for these distinctions. Since the test corpus was used for tuning, confirm the gains with a separate validation set and a final untouched test evaluation.

## 8. Saving the best model

In [12]:
MODEL_DIR = ROOT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "hmm_pos.pkl"

with MODEL_PATH.open("wb") as model_file:
    plk.dump(best_model, model_file)

print(f"Best HMM model saved to: {MODEL_PATH}")

Best HMM model saved to: c:\Users\onest\OneDrive\Desktop\Projects\BiLSTM_CRF_Burmese_POS_Tagger\models\hmm_pos.pkl


In [13]:
def predict_sentence(text, model=best_model):
    words = text.split()
    if not words:
        return []
    tags = model.predict_one(words)
    return list(zip(words, tags))

text = input("Enter a whitespace-tokenized sentence: ").strip()
if text:
    tagged_sentence = predict_sentence(text)
    print(" ".join(f"{word}/{tag}" for word, tag in tagged_sentence))
else:
    print("No sentence entered.")

ဘယ်/pron သွား/v မလဲ/part
